# MatGPT Telco 300M — guarded Colab training

This notebook builds a 306,226,176-parameter English and telecom base model. Run one stage at a time. Data, evidence, and checkpoints persist in Google Drive; temporary tokenizer and shard copies stay on the local runtime for speed.

## 1. Choose one stage

In [ ]:
RUN_STAGE = "prepare_data"  # @param ["prepare_data", "prepare", "smoke", "pilot", "full", "evaluate"]
DATA_PLAN = "pilot"  # @param ["pilot", "full"]
PREPARED_DATA_MODE = "prebuilt_shards"  # @param ["prebuilt_shards", "legacy_jsonl"]
ALLOW_FULL_DATA = False  # @param {type:"boolean"}
FULL_APPROVED = False  # @param {type:"boolean"}
GOOGLE_DRIVE_FREE_GB_OVERRIDE = 0.0  # @param {type:"number"}
PILOT_TOKENS = 20_000_000
SMOKE_MAX_STEPS = 20
SMOKE_RESUME_STEPS = 5
PILOT_MAX_ADDITIONAL_STEPS = 200
STAGES = {"prepare_data", "prepare", "smoke", "pilot", "full", "evaluate"}
assert RUN_STAGE in STAGES
assert DATA_PLAN in {"pilot", "full"}
assert PREPARED_DATA_MODE in {"prebuilt_shards", "legacy_jsonl"}
if RUN_STAGE in {"smoke", "pilot"}:
    assert DATA_PLAN == "pilot", f"{RUN_STAGE} requires DATA_PLAN='pilot'."
if RUN_STAGE == "full":
    assert DATA_PLAN == "full", "Full training requires DATA_PLAN='full'."
print(f"Selected stage={RUN_STAGE!r}, data plan={DATA_PLAN!r}")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Locate or clone the project

In [ ]:
import os
import subprocess
from pathlib import Path

PROJECT_DIR = Path("/content/train-llm-from-scratch")
REPOSITORY = "https://github.com/digotetso/train-llm-from-scratch.git"
if not (PROJECT_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", REPOSITORY, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only", "origin", "main"], check=True)
os.chdir(PROJECT_DIR)
print(subprocess.run(["git", "rev-parse", "HEAD"], check=True, text=True, capture_output=True).stdout.strip())

## 4. Install and authenticate

In [ ]:
import getpass
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
from huggingface_hub import login
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Hugging Face read token (input is hidden): " ).strip()
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
else:
    print("No token supplied. Public sources may still work, subject to Hugging Face access rules.")

## 5. Inspect the runtime

In [ ]:
import shutil
import torch

print("/content disk:", shutil.disk_usage("/content"))
print("Drive mount/cache filesystem:", shutil.disk_usage("/content/drive"))
print("Note: the Drive mount/cache value is not your Google account quota.")
print("torch:", torch.__version__, "cuda available:", torch.cuda.is_available())
if RUN_STAGE in {"prepare", "smoke", "pilot", "full", "evaluate"}:
    assert torch.cuda.is_available(), f"{RUN_STAGE} requires CUDA."
    properties = torch.cuda.get_device_properties(0)
    gpu_name = torch.cuda.get_device_name(0)
    print("gpu:", gpu_name, "memory GiB:", properties.total_memory / 1024**3)
    if properties.total_memory < 40 * 1024**3:
        print("Warning: less than 40 GiB VRAM; reduce micro-batch size and re-benchmark.")

## 6. Build fixed local and Drive paths

In [ ]:
import hashlib
import json
import yaml

DRIVE_ROOT = Path("/content/drive/MyDrive/matgpt_artifacts") / "matgpt_telco_300m"
CHECKED_CONFIG = PROJECT_DIR / "configs/matgpt_telco_300m.yaml"
SOURCE_REGISTRY = PROJECT_DIR / "configs/data/telco_300m_sources.yaml"
MIXTURE_CONFIG = PROJECT_DIR / "configs/data/telco_300m_mixture.yaml"
DATA_RECIPE_SHA256 = hashlib.sha256(b"telco-data-recipe-v1\0" + CHECKED_CONFIG.read_bytes() + b"\0" + SOURCE_REGISTRY.read_bytes() + b"\0" + MIXTURE_CONFIG.read_bytes()).hexdigest()
RECIPE_ROOT = DRIVE_ROOT / "recipes" / DATA_RECIPE_SHA256[:12]
WORK_ROOT = Path("/content/matgpt_work") / "matgpt_telco_300m" / DATA_RECIPE_SHA256[:12]
LEGACY_CORPUS_DIR = RECIPE_ROOT / "corpora" / DATA_PLAN
PREBUILT_CORPUS_DIR = WORK_ROOT / DATA_PLAN / "prebuilt"
CORPUS_DIR = PREBUILT_CORPUS_DIR if PREPARED_DATA_MODE == "prebuilt_shards" else LEGACY_CORPUS_DIR
EVAL_LITE_DIR = DRIVE_ROOT / "evaluation_assets/open_telco_lite"
EVAL_FULL_DIR = DRIVE_ROOT / "evaluation_assets/open_telco_full"
TOKENIZER_DIR = WORK_ROOT / DATA_PLAN / "tokenizer"
SHARD_DIR = PREBUILT_CORPUS_DIR if PREPARED_DATA_MODE == "prebuilt_shards" else WORK_ROOT / DATA_PLAN / "shards"
ARTIFACT_DRIVE_DIR = RECIPE_ROOT / "prepared" / DATA_PLAN
PILOT_TOKENIZER_DRIVE_DIR = RECIPE_ROOT / "prepared/pilot/tokenizer"
TOKENIZER_SELECTION_PATH = DRIVE_ROOT / "tokenizer_selection.json"
TOKENIZER_COMPARISON_PATH = DRIVE_ROOT / "comparison.json"
if PREPARED_DATA_MODE == "prebuilt_shards":
    from matgpt.tokenizer.candidate import validate_tokenizer_selection
    selected = json.loads(TOKENIZER_SELECTION_PATH.read_text(encoding="utf-8"))
    compared = json.loads(TOKENIZER_COMPARISON_PATH.read_text(encoding="utf-8"))
    SELECTED_TOKENIZER_SHA = validate_tokenizer_selection(selected, compared)
    COLAB_GATE_ROOT = DRIVE_ROOT / "evidence/tokenizers" / SELECTED_TOKENIZER_SHA
    COLAB_GATE_ROOT = COLAB_GATE_ROOT / DATA_PLAN / "colab"
    EVIDENCE_DIR = COLAB_GATE_ROOT
    RUN_DIR = COLAB_GATE_ROOT
else:
    SELECTED_TOKENIZER_SHA = None
    EVIDENCE_DIR = RECIPE_ROOT / "evidence" / DATA_PLAN
    RUN_DIR = RECIPE_ROOT / "runs" / DATA_PLAN
CONFIG_PATH = WORK_ROOT / DATA_PLAN / "config/matgpt_telco_300m.yaml"
for path in (WORK_ROOT, DRIVE_ROOT, RECIPE_ROOT, EVIDENCE_DIR, RUN_DIR, CONFIG_PATH.parent):
    path.mkdir(parents=True, exist_ok=True)

cfg = yaml.safe_load(CHECKED_CONFIG.read_text(encoding="utf-8"))
cfg["dataset"]["source_registry_path"] = str(SOURCE_REGISTRY)
cfg["dataset"]["mixture_config_path"] = str(MIXTURE_CONFIG)
cfg["dataset"]["normalized_dir"] = str(CORPUS_DIR)
cfg["tokenizer"]["output_dir"] = str(TOKENIZER_DIR)
cfg["tokenizer"]["probe_sets_path"] = str(PROJECT_DIR / "configs/data/telco_tokenizer_probes.yaml")
cfg["sharding"]["output_dir"] = str(SHARD_DIR)
cfg["run"]["output_dir"] = str(RUN_DIR)
if DATA_PLAN == "pilot":
    cfg["dataset"]["train_split"] = "pilot"
    cfg["dataset"]["training_splits"] = {"pilot": "pilot"}
    cfg["training"]["max_tokens"] = PILOT_TOKENS
    cfg["training"]["data_phases"] = [{"name": "pilot", "split": "pilot", "until_tokens": PILOT_TOKENS}]
    cfg["training"]["eval_interval_tokens"] = 5_000_000
    cfg["training"]["checkpoint_interval_tokens"] = 5_000_000
    cfg["training"]["sample_interval_tokens"] = 5_000_000
CONFIG_PATH.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
from matgpt.config import config_to_yaml, load_config
from matgpt.utils.hashing import sha256_file, sha256_text
CURRENT_CONFIG_SHA = sha256_text(config_to_yaml(load_config(CONFIG_PATH)))
print("Data recipe:", DATA_RECIPE_SHA256, "Config:", CONFIG_PATH, "Corpus:", CORPUS_DIR, "Run:", RUN_DIR)

## 7. Prepare isolated evaluation and training data

In [ ]:
def run_command(command):
    print("$", " ".join(map(str, command)))
    result = subprocess.run(list(map(str, command)), text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(map(str, command))}")
    return result

from matgpt.data.telco_prepare import corpus_has_exact_token_quotas
from matgpt.storage import google_drive_storage_evidence, operator_storage_evidence, require_free_storage_gib

def build_corpus_command(stages, *, tokenizer_dir=None, force=False):
    command = [sys.executable, "scripts/prepare_telco_corpus.py", "--sources", SOURCE_REGISTRY, "--mixture", MIXTURE_CONFIG]
    for stage in stages:
        command += ["--stage", stage]
    command += ["--output-dir", CORPUS_DIR]
    for pattern_path in sorted(EVAL_LITE_DIR.glob("*.jsonl")) + sorted(EVAL_FULL_DIR.glob("*.jsonl")):
        command += ["--contamination-patterns", pattern_path]
    if DATA_PLAN == "pilot":
        command += ["--total-tokens", str(PILOT_TOKENS)]
    else:
        command += ["--allow-full-data"]
    if tokenizer_dir is not None:
        command += ["--tokenizer-dir", tokenizer_dir]
    if force:
        command += ["--force"]
    return command

if RUN_STAGE == "prepare_data" and PREPARED_DATA_MODE == "prebuilt_shards":
    for dataset_name, destination in (("lite", EVAL_LITE_DIR), ("full", EVAL_FULL_DIR)):
        if not (destination / "manifest.json").is_file():
            run_command([sys.executable, "scripts/prepare_open_telco_evals.py", "--sources", SOURCE_REGISTRY, "--dataset", dataset_name, "--output-dir", destination])
    print("Prebuilt mode selected: corpus collection and tokenizer fitting stay on the Mac. Next choose RUN_STAGE='prepare'.")
if RUN_STAGE == "prepare_data" and PREPARED_DATA_MODE == "legacy_jsonl":
    for dataset_name, destination in (("lite", EVAL_LITE_DIR), ("full", EVAL_FULL_DIR)):
        if not (destination / "manifest.json").is_file():
            run_command([sys.executable, "scripts/prepare_open_telco_evals.py", "--sources", SOURCE_REGISTRY, "--dataset", dataset_name, "--output-dir", destination])
    stages = ["pilot"] if DATA_PLAN == "pilot" else ["main", "cooldown"]
    if DATA_PLAN == "full":
        assert ALLOW_FULL_DATA, "Set ALLOW_FULL_DATA=True to authorize the 12B-token corpus build."
        assert (PILOT_TOKENIZER_DRIVE_DIR / "tokenizer.json").is_file() and (PILOT_TOKENIZER_DRIVE_DIR / "special_tokens.json").is_file(), "Run the pilot prepare stage first; full data must use its frozen tokenizer."
        try:
            from google.colab import auth as colab_auth
            from googleapiclient.discovery import build as build_google_api
            colab_auth.authenticate_user()
            drive_about = build_google_api("drive", "v3", cache_discovery=False).about().get(fields="storageQuota").execute()
            drive_storage = google_drive_storage_evidence(drive_about)
        except Exception as exc:
            if GOOGLE_DRIVE_FREE_GB_OVERRIDE <= 0:
                raise RuntimeError("Could not read the Google Drive account quota. Set GOOGLE_DRIVE_FREE_GB_OVERRIDE from the Drive Storage page and rerun.") from exc
            drive_storage = operator_storage_evidence(GOOGLE_DRIVE_FREE_GB_OVERRIDE)
        require_free_storage_gib(drive_storage, 140.0)
        (EVIDENCE_DIR / "drive_storage.json").write_text(json.dumps(drive_storage, indent=2, sort_keys=True) + "\n", encoding="utf-8")
        print("Google Drive account quota evidence:", drive_storage)
    plan_paths = []
    for stage in stages:
        plan_path = EVIDENCE_DIR / f"mixture_plan_{stage}.json"
        command = [sys.executable, "scripts/plan_telco_mixture.py", "--sources", SOURCE_REGISTRY, "--mixture", MIXTURE_CONFIG, "--stage", stage, "--output", plan_path]
        if stage == "pilot":
            command += ["--total-tokens", str(PILOT_TOKENS)]
        run_command(command)
        plan_paths.append(plan_path)
    plans = [json.loads(path.read_text(encoding="utf-8")) for path in plan_paths]
    corpus_manifest_exists = (CORPUS_DIR / "manifest.json").is_file()
    corpus_ready = corpus_manifest_exists
    if DATA_PLAN == "full":
        corpus_ready = corpus_has_exact_token_quotas(CORPUS_DIR, PILOT_TOKENIZER_DRIVE_DIR, plans)
    if not corpus_ready:
        tokenizer_for_quota = PILOT_TOKENIZER_DRIVE_DIR if DATA_PLAN == "full" else None
        run_command(build_corpus_command(stages, tokenizer_dir=tokenizer_for_quota, force=corpus_manifest_exists))
    print("Data preparation complete. Next choose RUN_STAGE='prepare'.")
elif RUN_STAGE != "prepare_data":
    print("Skipped: this cell acts only when RUN_STAGE='prepare_data'.")

## 8. Prepare tokenizer and shards

In [ ]:
def atomic_snapshot(source, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    staging = destination.with_name(destination.name + ".staging")
    backup = destination.with_name(destination.name + ".previous")
    if staging.exists():
        shutil.rmtree(staging)
    shutil.copytree(source, staging)
    if backup.exists():
        shutil.rmtree(backup)
    if destination.exists():
        destination.replace(backup)
    staging.replace(destination)

def tokenizer_artifact_sha256(path):
    try:
        metadata = json.loads((path / "special_tokens.json").read_text(encoding="utf-8"))
        actual = sha256_file(path / "tokenizer.json")
    except (OSError, KeyError, json.JSONDecodeError):
        return None
    return actual if metadata.get("tokenizer_sha256") == actual else None

def restore_prebuilt_shards(*, runtime_root, drive_root, source_corpus_dir, source_tokenizer_dir, target_corpus_dir, target_tokenizer_dir, selection_path, comparison_path, pilot_refresh_path, expected_fingerprints, expected_splits, require_pilot_gates, pilot_gate_validator=None, pilot_reuse_validator=None):
    from pathlib import Path, PurePosixPath
    from matgpt.data.shard import resolve_shard_artifact_path
    from matgpt.tokenizer.candidate import validate_tokenizer_selection
    from matgpt.tokenizer.io import load_tokenizer_metadata
    from matgpt.training.dataset import load_verified_shard_metadata
    from matgpt.utils.hashing import sha256_file, sha256_json
    from matgpt.utils.paths import require_managed_path

    runtime_root = Path(runtime_root)
    runtime_root.mkdir(parents=True, exist_ok=True)
    runtime_root = require_managed_path(runtime_root, runtime_root, kind="directory", allow_missing=False)
    try:
        target_corpus_dir = require_managed_path(runtime_root, Path(target_corpus_dir), allow_missing=True)
        target_tokenizer_dir = require_managed_path(runtime_root, Path(target_tokenizer_dir), allow_missing=True)
        managed_targets = (target_corpus_dir, target_tokenizer_dir)
        if runtime_root in managed_targets or target_corpus_dir == target_tokenizer_dir or target_corpus_dir.is_relative_to(target_tokenizer_dir) or target_tokenizer_dir.is_relative_to(target_corpus_dir):
            raise ValueError("runtime restore targets must be distinct descendants")
        for target in managed_targets:
            require_managed_path(runtime_root, target.with_name(target.name + ".restore-staging"), allow_missing=True)
    except ValueError as error:
        raise ValueError(f"runtime root containment failure: {error}") from error
    drive_root = require_managed_path(Path(drive_root), Path(drive_root), kind="directory", allow_missing=False)
    source_corpus_dir = require_managed_path(drive_root, Path(source_corpus_dir), kind="directory", allow_missing=False)
    source_tokenizer_dir = require_managed_path(drive_root, Path(source_tokenizer_dir), kind="directory", allow_missing=False)
    selection_path = require_managed_path(drive_root, Path(selection_path), kind="file", allow_missing=False)
    comparison_path = require_managed_path(drive_root, Path(comparison_path), kind="file", allow_missing=False)
    pilot_refresh_path = require_managed_path(drive_root, Path(pilot_refresh_path), kind="file", allow_missing=False)

    selection = json.loads(selection_path.read_text(encoding="utf-8"))
    comparison = json.loads(comparison_path.read_text(encoding="utf-8"))
    selected_sha = validate_tokenizer_selection(selection, comparison)
    expected_plan = "pilot" if tuple(expected_splits) == ("pilot", "validation") else "full" if tuple(expected_splits) == ("main", "cooldown", "validation") else None
    if expected_plan is None:
        raise ValueError("prebuilt split plan is invalid")
    if selection_path != drive_root / "tokenizer_selection.json" or comparison_path != drive_root / "comparison.json":
        raise ValueError("selected tokenizer workflow path mismatch")
    if pilot_refresh_path != drive_root / "evidence/tokenizers" / selected_sha / "pilot/pilot_refresh.json":
        raise ValueError("pilot refresh path does not match selected tokenizer")
    if source_corpus_dir != drive_root / "corpora" / expected_plan / selected_sha:
        raise ValueError("final corpus path does not match selected tokenizer and plan")
    tokenizer_json = require_managed_path(source_tokenizer_dir, source_tokenizer_dir / "tokenizer.json", kind="file", allow_missing=False)
    require_managed_path(source_tokenizer_dir, source_tokenizer_dir / "special_tokens.json", kind="file", allow_missing=False)
    if sha256_file(tokenizer_json) != selected_sha or load_tokenizer_metadata(source_tokenizer_dir).get("tokenizer_sha256") != selected_sha:
        raise ValueError("selected tokenizer fingerprint mismatch")

    pilot_refresh = json.loads(pilot_refresh_path.read_text(encoding="utf-8"))
    pilot_refresh_unsigned = dict(pilot_refresh)
    pilot_refresh_hash = pilot_refresh_unsigned.pop("pilot_refresh_sha256", None)
    if pilot_refresh_hash != sha256_json(pilot_refresh_unsigned):
        raise ValueError("pilot refresh checksum mismatch")
    if pilot_refresh.get("status") != "ready_for_colab":
        raise ValueError("pilot refresh status is not ready_for_colab")
    if pilot_refresh.get("action") not in {"reuse", "rebuild"}:
        raise ValueError("pilot refresh action is invalid")
    if pilot_refresh.get("selected_tokenizer_sha256") != selected_sha:
        raise ValueError("pilot refresh does not match selected tokenizer")
    if pilot_refresh.get("selection_file_sha256") != sha256_file(selection_path) or pilot_refresh.get("comparison_file_sha256") != sha256_file(comparison_path):
        raise ValueError("pilot refresh workflow fingerprints changed")
    if pilot_refresh.get("winner") != selection.get("winner") or pilot_refresh.get("selection_comparison_sha256") != selection.get("comparison_sha256") or pilot_refresh.get("comparison_sha256") != comparison.get("comparison_sha256"):
        raise ValueError("pilot refresh selected workflow fingerprint mismatch")
    pilot_gates_validated = False
    if require_pilot_gates:
        if pilot_refresh.get("action") == "reuse" and pilot_refresh.get("refreshed_pilot_gates_passed") is True and pilot_refresh.get("pending_colab_gates") == []:
            if pilot_reuse_validator is None:
                from scripts.prepare_telco_local import _pilot_reuse_evidence as pilot_reuse_validator
            if pilot_refresh.get("artifacts") != pilot_reuse_validator(drive_root, selected_sha):
                raise ValueError("pilot refresh recorded artifact fingerprints changed")
            pilot_gates_validated = True
        elif pilot_refresh.get("action") == "rebuild" and pilot_refresh.get("refreshed_pilot_gates_passed") is False and pilot_refresh.get("pending_colab_gates") == ["smoke", "pilot", "evaluation"]:
            pilot_build_identity = pilot_refresh.get("build_identity_sha256")
            if not isinstance(pilot_build_identity, str) or len(pilot_build_identity) != 64 or any(char not in "0123456789abcdef" for char in pilot_build_identity):
                raise ValueError("pilot refresh build identity is invalid")
            if pilot_gate_validator is None:
                from scripts.prepare_telco_local import _pilot_colab_evidence as pilot_gate_validator
            gate_root = drive_root / "evidence/tokenizers" / selected_sha / "pilot/colab"
            current_gates = pilot_gate_validator(drive_dir=drive_root, gate_root=gate_root, evaluation_root=gate_root / "evaluation", tokenizer_sha256=selected_sha, build_identity_sha256=pilot_build_identity)
            if not isinstance(current_gates, dict) or not current_gates:
                raise ValueError("pilot refresh gates do not match selected tokenizer")
            pilot_gates_validated = True
        else:
            raise ValueError("pilot refresh gates do not match selected tokenizer")

    manifest_path = source_corpus_dir / "manifest.json"
    if not manifest_path.is_file() or manifest_path.is_symlink():
        raise AssertionError("final corpus manifest is incomplete")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    manifest_unsigned = dict(manifest)
    manifest_hash = manifest_unsigned.pop("manifest_sha256", None)
    if manifest_hash != sha256_json(manifest_unsigned):
        raise ValueError("final corpus manifest checksum mismatch")
    if manifest.get("complete") is not True or manifest.get("status") != "complete":
        raise AssertionError("final corpus manifest is incomplete")
    if manifest.get("version") != 2 or manifest.get("builder") != "local_corpus" or manifest.get("storage_format") != "chunked_prebuilt_v1":
        raise ValueError("final corpus manifest builder/schema mismatch")
    fingerprints = manifest.get("fingerprints")
    if not isinstance(fingerprints, dict) or manifest.get("build_identity_sha256") != sha256_json(fingerprints):
        raise ValueError("final corpus build fingerprint mismatch")
    if fingerprints.get("tokenizer_sha256") != selected_sha or manifest.get("quota_counting", {}).get("tokenizer_sha256") != selected_sha:
        raise ValueError("selected tokenizer fingerprint mismatch")
    if expected_plan == "pilot" and pilot_refresh.get("action") == "rebuild" and pilot_refresh.get("build_identity_sha256") != manifest.get("build_identity_sha256"):
        raise ValueError("pilot refresh build identity does not match final pilot corpus")
    for name, expected in expected_fingerprints.items():
        if fingerprints.get(name) != expected:
            raise ValueError(f"final corpus {name} mismatch")
    logical_keys = ("version", "builder", "storage_format", "build_identity_sha256", "fingerprints", "stages", "sources", "split_stats", "breakdowns", "unit_artifacts", "audits")
    if manifest.get("content_sha256") != sha256_json({name: manifest.get(name) for name in logical_keys}):
        raise ValueError("final corpus content fingerprint mismatch")

    def safe_relative(value):
        if not isinstance(value, str) or not value or "\\" in value:
            raise ValueError("artifact path must be a safe relative path")
        relative = PurePosixPath(value)
        if relative.is_absolute() or ".." in relative.parts or str(relative) != value:
            raise ValueError("artifact path must be a safe relative path")
        return Path(*relative.parts)

    split_stats = manifest.get("split_stats")
    if not isinstance(split_stats, dict):
        raise ValueError("final corpus split evidence is missing")
    for split in expected_splits:
        stats = split_stats.get(split)
        raw_chunks = stats.get("raw_chunks") if isinstance(stats, dict) else None
        if not isinstance(raw_chunks, list) or not raw_chunks:
            raise ValueError(f"final corpus {split} raw chunk evidence is missing")
        for chunk in raw_chunks:
            if not isinstance(chunk, dict):
                raise ValueError("final raw chunk evidence is invalid")
            safe_relative(chunk.get("path"))
            chunk_size, chunk_sha = chunk.get("size"), chunk.get("sha256")
            if type(chunk_size) is not int or chunk_size < 0 or not isinstance(chunk_sha, str) or len(chunk_sha) != 64 or any(char not in "0123456789abcdef" for char in chunk_sha):
                raise ValueError("final raw chunk fingerprint evidence is invalid")

    verified_files = {}
    def verify_record(record, *, internal_hash_field=None):
        if not isinstance(record, dict):
            raise ValueError("final artifact evidence is missing")
        relative = safe_relative(record.get("path"))
        path = require_managed_path(source_corpus_dir, source_corpus_dir / relative, kind="file", allow_missing=False)
        if path.stat().st_size != record.get("size") or sha256_file(path) != record.get("sha256"):
            raise ValueError(f"final artifact fingerprint mismatch: {relative}")
        if internal_hash_field is not None:
            payload = json.loads(path.read_text(encoding="utf-8"))
            unsigned = dict(payload)
            stored = unsigned.pop(internal_hash_field, None)
            if stored != sha256_json(unsigned):
                raise ValueError(f"final artifact internal fingerprint mismatch: {relative}")
        verified_files[relative.as_posix()] = path
        return path

    audits = manifest.get("audits")
    if not isinstance(audits, dict):
        raise ValueError("final corpus audit evidence is missing")
    for name in ("quota_audit", "license_audit", "quality_audit", "overlap_audit"):
        verify_record(audits.get(name), internal_hash_field="audit_sha256")
    verify_record(manifest.get("calibration_report"), internal_hash_field="calibration_report_sha256")

    split_metadata = manifest.get("split_metadata")
    if not isinstance(split_metadata, dict):
        raise ValueError("final corpus split metadata is missing")
    shard_sources = {}
    for split in expected_splits:
        record = split_metadata.get(split)
        metadata_path = verify_record(record, internal_hash_field="metadata_sha256")
        _, metadata = load_verified_shard_metadata(metadata_path, metadata_root=source_corpus_dir, finalized_root=source_corpus_dir, finalized_artifact=record, require_internal_fingerprint=True)
        if metadata.get("split") != split or metadata.get("tokenizer_sha256") != selected_sha or metadata.get("dtype") != "uint16" or metadata.get("append_eos") is not True:
            raise ValueError(f"{split} shard metadata fingerprint/dtype/EOS mismatch")
        for shard in metadata.get("shards", []):
            shard_path = resolve_shard_artifact_path(metadata_path, shard.get("path"), shard_root=source_corpus_dir)
            expected_bytes = int(shard.get("num_tokens", -1)) * 2
            if shard_path.stat().st_size != expected_bytes or shard.get("byte_size") != expected_bytes or sha256_file(shard_path) != shard.get("sha256"):
                raise ValueError(f"{split} shard fingerprint mismatch: {shard.get('path')}")
            shard_sources[safe_relative(shard.get("path")).as_posix()] = shard_path

    def source_tree(path):
        sources = {}
        for source in path.rglob("*"):
            if source.is_symlink():
                raise ValueError(f"runtime source tree contains a symbolic link: {source}")
            if source.is_file():
                sources[safe_relative(source.relative_to(path).as_posix()).as_posix()] = source
        if not sources:
            raise ValueError("runtime source tree has no files")
        return sources

    def tree_matches(target, sources):
        if not target.is_dir() or target.is_symlink():
            return False
        observed = {}
        for path in target.rglob("*"):
            if path.is_symlink():
                return False
            if path.is_file():
                observed[path.relative_to(target).as_posix()] = path
        return set(observed) == set(sources) and all(sha256_file(observed[relative]) == sha256_file(source) for relative, source in sources.items())

    def install_runtime_tree(target, sources, label):
        if target.exists():
            if tree_matches(target, sources):
                return
            raise ValueError(f"existing runtime {label} differs from verified source; start a clean runtime")
        staging = target.with_name(target.name + ".restore-staging")
        if staging.exists():
            if not tree_matches(staging, sources):
                raise ValueError(f"incomplete runtime {label} staging tree; start a clean runtime")
        else:
            staging.mkdir(parents=True)
            for relative, source in sources.items():
                destination = staging / safe_relative(relative)
                destination.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(source, destination)
            if not tree_matches(staging, sources):
                raise ValueError(f"runtime {label} copy fingerprint mismatch")
        staging.replace(target)

    tokenizer_sources = source_tree(source_tokenizer_dir)
    corpus_sources = {**verified_files, **shard_sources, "manifest.json": manifest_path}
    install_runtime_tree(target_tokenizer_dir, tokenizer_sources, "tokenizer")
    install_runtime_tree(target_corpus_dir, corpus_sources, "corpus")
    return {"tokenizer_sha256": selected_sha, "build_identity_sha256": manifest["build_identity_sha256"], "tokenization_invoked": False, "pilot_gates_validated": pilot_gates_validated, "shard_count": len(shard_sources)}

def restore_current_prebuilt():
    from dataclasses import asdict
    from matgpt.data.contamination import pattern_fingerprint
    from matgpt.data.mixture import build_mixture_plan, load_mixture_config
    from matgpt.data.quality import load_contamination_patterns
    from matgpt.data.sources import load_source_registry
    from matgpt.tokenizer.candidate import load_tokenizer_candidate_config, validate_tokenizer_selection
    from matgpt.utils.hashing import sha256_json
    selection = json.loads(TOKENIZER_SELECTION_PATH.read_text(encoding="utf-8"))
    comparison = json.loads(TOKENIZER_COMPARISON_PATH.read_text(encoding="utf-8"))
    selected_sha = validate_tokenizer_selection(selection, comparison)
    candidate_recipe = load_tokenizer_candidate_config(PROJECT_DIR / "configs/data/telco_300m_tokenizer_candidate.yaml")
    source_tokenizer_dir = PILOT_TOKENIZER_DRIVE_DIR if selection["winner"] == candidate_recipe.baseline_label else DRIVE_ROOT / "tokenizers" / candidate_recipe.candidate_label
    registry = load_source_registry(SOURCE_REGISTRY)
    mixture = load_mixture_config(MIXTURE_CONFIG)
    stages = ["pilot"] if DATA_PLAN == "pilot" else ["main", "cooldown"]
    plans = [build_mixture_plan(registry, mixture, stage) for stage in stages]
    contamination_paths = sorted(EVAL_LITE_DIR.glob("*.jsonl")) + sorted(EVAL_FULL_DIR.glob("*.jsonl"))
    expected_fingerprints = {"source_registry_sha256": sha256_json(asdict(registry)), "plan_sha256": sha256_json(plans), "contamination_sha256": pattern_fingerprint(load_contamination_patterns(contamination_paths))}
    return restore_prebuilt_shards(runtime_root=WORK_ROOT, drive_root=DRIVE_ROOT, source_corpus_dir=DRIVE_ROOT / "corpora" / DATA_PLAN / selected_sha, source_tokenizer_dir=source_tokenizer_dir, target_corpus_dir=CORPUS_DIR, target_tokenizer_dir=TOKENIZER_DIR, selection_path=TOKENIZER_SELECTION_PATH, comparison_path=TOKENIZER_COMPARISON_PATH, pilot_refresh_path=DRIVE_ROOT / "evidence/tokenizers" / selected_sha / "pilot/pilot_refresh.json", expected_fingerprints=expected_fingerprints, expected_splits=tuple(stages + ["validation"]), require_pilot_gates=DATA_PLAN == "full")

if RUN_STAGE == "prepare" and PREPARED_DATA_MODE == "prebuilt_shards":
    if DATA_PLAN == "full":
        local_free = shutil.disk_usage("/content").free
        assert local_free >= 35 * 1024**3, f"Full preparation needs at least 35 GiB free under /content; observed {local_free / 1024**3:.1f} GiB."
    PREBUILT_RESTORE = restore_current_prebuilt()
    shutil.copy2(CONFIG_PATH, EVIDENCE_DIR / "config.yaml")
    run_command([sys.executable, "scripts/preflight_t4.py", "--config", CONFIG_PATH, "--report-path", EVIDENCE_DIR / "preflight.json", "--require-supported-gpu", "--min-free-disk-gb", "0"] )
    benchmark = run_command([sys.executable, "scripts/benchmark_t4.py", "--config", CONFIG_PATH, "--batch-sizes", "4,8,12", "--steps", "3"] )
    benchmark_payload = json.loads(benchmark.stdout)
    benchmark_payload["config_sha256"] = CURRENT_CONFIG_SHA
    configured_batch = next(row for row in benchmark_payload["results"] if row["batch_size"] == 8)
    assert configured_batch["status"] == "ok", f"Configured batch 8 failed: {configured_batch}"
    assert configured_batch["memory_fraction"] <= 0.90, f"Configured batch 8 has insufficient memory headroom: {configured_batch}"
    (EVIDENCE_DIR / "benchmark.json").write_text(json.dumps(benchmark_payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    print("Fingerprint-verified prebuilt artifacts restored. Next choose RUN_STAGE='smoke'.")
legacy_prepare = False
if PREPARED_DATA_MODE == "legacy_jsonl":
    legacy_prepare = RUN_STAGE == "prepare"
if legacy_prepare:
    assert (CORPUS_DIR / "manifest.json").is_file(), "Run prepare_data first."
    if DATA_PLAN == "full":
        local_free = shutil.disk_usage("/content").free
        assert local_free >= 35 * 1024**3, f"Full preparation needs at least 35 GiB free under /content; observed {local_free / 1024**3:.1f} GiB."
    stages = ["pilot"] if DATA_PLAN == "pilot" else ["main", "cooldown"]
    plan_paths = [EVIDENCE_DIR / f"mixture_plan_{stage}.json" for stage in stages]
    plans = [json.loads(path.read_text(encoding="utf-8")) for path in plan_paths]
    frozen_tokenizer_dir = PILOT_TOKENIZER_DRIVE_DIR
    frozen_tokenizer_sha256 = tokenizer_artifact_sha256(frozen_tokenizer_dir)
    local_tokenizer_sha256 = tokenizer_artifact_sha256(TOKENIZER_DIR)
    if frozen_tokenizer_sha256 is not None:
        if local_tokenizer_sha256 != frozen_tokenizer_sha256:
            atomic_snapshot(frozen_tokenizer_dir, TOKENIZER_DIR)
    elif DATA_PLAN == "full":
        raise AssertionError("Run the pilot prepare stage first; its frozen tokenizer is required for full preparation.")
    else:
        if local_tokenizer_sha256 is None:
            run_command([sys.executable, "scripts/train_tokenizer.py", "--config", CONFIG_PATH])
            local_tokenizer_sha256 = tokenizer_artifact_sha256(TOKENIZER_DIR)
            assert local_tokenizer_sha256 is not None, "Tokenizer training produced invalid artifacts."
        atomic_snapshot(TOKENIZER_DIR, frozen_tokenizer_dir)
        frozen_tokenizer_sha256 = local_tokenizer_sha256
    print("Frozen tokenizer:", frozen_tokenizer_sha256)
    if not corpus_has_exact_token_quotas(CORPUS_DIR, TOKENIZER_DIR, plans):
        run_command(build_corpus_command(stages, tokenizer_dir=TOKENIZER_DIR, force=True))
    assert corpus_has_exact_token_quotas(CORPUS_DIR, TOKENIZER_DIR, plans), "Corpus is not bound to the frozen tokenizer and current plans."
    audit_command = [sys.executable, "scripts/audit_telco_corpus.py", "--tokenizer-dir", TOKENIZER_DIR, "--tolerance", "0.03", "--output", EVIDENCE_DIR / "quota_audit.json"]
    for stage in stages:
        audit_command += ["--input", CORPUS_DIR / f"{stage}.jsonl", "--plan", EVIDENCE_DIR / f"mixture_plan_{stage}.json"]
    run_command(audit_command)
    run_command([sys.executable, "scripts/tokenize_and_shard.py", "--config", CONFIG_PATH])
    run_command([sys.executable, "scripts/preflight_t4.py", "--config", CONFIG_PATH, "--report-path", EVIDENCE_DIR / "preflight.json", "--require-supported-gpu", "--min-free-disk-gb", "0"] )
    benchmark = run_command([sys.executable, "scripts/benchmark_t4.py", "--config", CONFIG_PATH, "--batch-sizes", "4,8,12", "--steps", "3"] )
    benchmark_payload = json.loads(benchmark.stdout)
    benchmark_payload["config_sha256"] = CURRENT_CONFIG_SHA
    configured_batch = next(row for row in benchmark_payload["results"] if row["batch_size"] == 8)
    assert configured_batch["status"] == "ok", f"Configured batch 8 failed: {configured_batch}"
    assert configured_batch["memory_fraction"] <= 0.90, f"Configured batch 8 has insufficient memory headroom: {configured_batch}"
    (EVIDENCE_DIR / "benchmark.json").write_text(json.dumps(benchmark_payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    if DATA_PLAN == "full":
        atomic_snapshot(TOKENIZER_DIR, ARTIFACT_DRIVE_DIR / "tokenizer")
    atomic_snapshot(SHARD_DIR, ARTIFACT_DRIVE_DIR / "shards")
    shutil.copy2(CONFIG_PATH, ARTIFACT_DRIVE_DIR / "config.yaml")
    print("Artifact preparation complete. Next choose RUN_STAGE='smoke'.")
elif RUN_STAGE != "prepare":
    print("Skipped: this cell acts only when RUN_STAGE='prepare'.")

## 9. Verify evidence gates

In [ ]:
def restore_prepared_artifacts():
    for name, local in (("tokenizer", TOKENIZER_DIR), ("shards", SHARD_DIR)):
        saved = ARTIFACT_DRIVE_DIR / name
        assert saved.is_dir(), f"Missing prepared {name}: {saved}"
        if not local.exists():
            local.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(saved, local)

def require_full_training_approval(full_approved):
    assert full_approved, "Set FULL_APPROVED=True only after reviewing pilot evidence."

if RUN_STAGE in {"smoke", "pilot", "full", "evaluate"}:
    for evidence_name in ("preflight.json", "benchmark.json"):
        assert (EVIDENCE_DIR / evidence_name).is_file(), f"Missing evidence: {evidence_name}"
    if PREPARED_DATA_MODE == "prebuilt_shards":
        PREBUILT_RESTORE = restore_current_prebuilt()
        quota_audit_path = CORPUS_DIR / "quota_audit.json"
    else:
        restore_prepared_artifacts()
        quota_audit_path = EVIDENCE_DIR / "quota_audit.json"
    assert quota_audit_path.is_file(), f"Missing evidence: {quota_audit_path}"
    assert json.loads(quota_audit_path.read_text())["passed"] is True
    prepared_preflight = json.loads((EVIDENCE_DIR / "preflight.json").read_text())
    assert prepared_preflight["status"] == "pass"
    prepared_config = next(check for check in prepared_preflight["checks"] if check["name"] == "config")
    assert prepared_config["details"]["config_sha256"] == CURRENT_CONFIG_SHA, "Prepared preflight belongs to a different config."
    assert json.loads((EVIDENCE_DIR / "benchmark.json").read_text())["config_sha256"] == CURRENT_CONFIG_SHA, "Benchmark belongs to a different config."
    run_command([sys.executable, "scripts/preflight_t4.py", "--config", CONFIG_PATH, "--report-path", EVIDENCE_DIR / f"preflight_{RUN_STAGE}.json", "--require-supported-gpu", "--min-free-disk-gb", "0"] )
if RUN_STAGE == "pilot":
    assert (EVIDENCE_DIR / "smoke_resume_verified.json").is_file(), "Run smoke first."
if RUN_STAGE == "full":
    require_full_training_approval(FULL_APPROVED)
    if PREPARED_DATA_MODE == "legacy_jsonl":
        pilot_gate = RECIPE_ROOT / "evidence/pilot/pilot_complete.json"
        assert pilot_gate.is_file(), f"Missing completed pilot gate: {pilot_gate}"
print("Evidence gates checked for", RUN_STAGE)

## 10. Run the selected stage

In [ ]:
from matgpt.training.checkpoint_provenance import snapshot_checkpoint
from matgpt.utils.hashing import sha256_json
pilot_manifest_path = CORPUS_DIR / "manifest.json"
pilot_manifest = json.loads(pilot_manifest_path.read_text(encoding="utf-8"))
pilot_tokenizer_sha = json.loads((TOKENIZER_DIR / "special_tokens.json").read_text(encoding="utf-8"))["tokenizer_sha256"]
pilot_build_identity = pilot_manifest.get("build_identity_sha256") or sha256_json({"format": "legacy_telco_prepare_v1", "manifest_sha256": pilot_manifest["manifest_sha256"], "tokenizer_sha256": pilot_tokenizer_sha})
artifact_identity = {"config_sha256": CURRENT_CONFIG_SHA, "tokenizer_sha256": pilot_tokenizer_sha, "dataset_manifest_sha256": sha256_file(pilot_manifest_path), "dataset_manifest_identity_sha256": pilot_manifest["manifest_sha256"], "build_identity_sha256": pilot_build_identity}
SMOKE_MAX_STEPS = 20
SMOKE_RESUME_STEPS = 5
checkpoint_dir = RUN_DIR / "checkpoints"
latest = checkpoint_dir / "latest.pt"
if RUN_STAGE == "smoke":
    command = [sys.executable, "scripts/pretrain.py", "--config", CONFIG_PATH, "--max-steps", str(SMOKE_MAX_STEPS)]
    if latest.is_file():
        command += ["--resume-from", latest]
    run_command(command)
    run_command([sys.executable, "scripts/pretrain.py", "--config", CONFIG_PATH, "--resume-from", latest, "--verify-only"] )
    run_command([sys.executable, "scripts/pretrain.py", "--config", CONFIG_PATH, "--resume-from", latest, "--max-steps", str(SMOKE_RESUME_STEPS)] )
    torch.save(torch.load(latest, map_location="cpu", weights_only=False)["state"], EVIDENCE_DIR / "smoke_resume_state.pt")
    smoke_binding = snapshot_checkpoint(latest, checkpoint_dir, label="smoke")
    (EVIDENCE_DIR / "smoke_resume_verified.json").write_text(json.dumps({"status": "pass", "resume_verified": True, "checkpoint": smoke_binding["path"], "checkpoint_binding": smoke_binding, "artifact_identity": artifact_identity}, indent=2, sort_keys=True) + "\n")
elif RUN_STAGE == "pilot":
    assert latest.is_file(), "Smoke checkpoint is missing."
    run_command([sys.executable, "scripts/pretrain.py", "--config", CONFIG_PATH, "--resume-from", latest, "--max-steps", str(PILOT_MAX_ADDITIONAL_STEPS)] )
    state = torch.load(latest, map_location="cpu", weights_only=False)["state"]
    assert int(state["tokens_processed"]) >= PILOT_TOKENS, "Pilot did not reach its token target."
    pilot_binding = snapshot_checkpoint(latest, checkpoint_dir, label="pilot-latest")
    best = checkpoint_dir / "best.pt"
    assert best.is_file(), "Pilot best checkpoint is missing."
    pilot_best_binding = snapshot_checkpoint(best, checkpoint_dir, label="pilot-best")
    smoke_binding = json.loads((EVIDENCE_DIR / "smoke_resume_verified.json").read_text(encoding="utf-8"))["checkpoint_binding"]
    assert smoke_binding != pilot_binding, "Smoke and pilot snapshots must be distinct."
    (EVIDENCE_DIR / "pilot_complete.json").write_text(json.dumps({"status": "pass", "complete": True, "tokens_processed": int(state["tokens_processed"]), "checkpoint": pilot_binding["path"], "checkpoint_binding": pilot_binding, "checkpoint_bindings": [pilot_binding, pilot_best_binding], "artifact_identity": artifact_identity}, indent=2, sort_keys=True) + "\n")
elif RUN_STAGE == "full":
    assert FULL_APPROVED
    command = [sys.executable, "scripts/pretrain.py", "--config", CONFIG_PATH]
    if latest.is_file():
        command += ["--resume-from", latest]
    run_command(command)
else:
    print("No training started for this stage.")

## 11. Evaluate checkpoints

In [ ]:
if RUN_STAGE == "evaluate":
    from datetime import datetime, timezone
    checkpoint_dir = RUN_DIR / "checkpoints"
    from matgpt.training.checkpoint_provenance import checkpoint_binding
    pilot_gate = json.loads((EVIDENCE_DIR / "pilot_complete.json").read_text(encoding="utf-8"))
    selected_pilot_checkpoint = Path(pilot_gate["checkpoint_binding"]["path"])
    assert checkpoint_binding(selected_pilot_checkpoint) == pilot_gate["checkpoint_binding"], "Pilot checkpoint changed; rerun pilot gates."
    candidates = [selected_pilot_checkpoint] + sorted(checkpoint_dir.glob("pilot-*.pt"))
    checkpoints = []
    for checkpoint in candidates:
        if checkpoint.is_file() and checkpoint not in checkpoints:
            checkpoints.append(checkpoint)
    assert checkpoints, f"No checkpoints found in {checkpoint_dir}"
    evaluation_dir = RUN_DIR / "evaluation" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    evaluation_dir.mkdir(parents=True, exist_ok=False)
    task_dir = EVAL_FULL_DIR if (EVAL_FULL_DIR / "manifest.json").is_file() else EVAL_LITE_DIR
    tasks = sorted(task_dir.glob("*.jsonl"))
    for index, checkpoint in enumerate(checkpoints):
        label = f"checkpoint_{index:02d}_{checkpoint.stem}"
        run_command([sys.executable, "scripts/evaluate.py", "--config", CONFIG_PATH, "--checkpoint", checkpoint, "--output", evaluation_dir / f"{label}_base.json"] )
        task_command = [sys.executable, "scripts/evaluate_tasks.py", "--config", CONFIG_PATH, "--checkpoint", checkpoint, "--output", evaluation_dir / f"{label}_open_telco.json"]
        for task in tasks:
            task_command += ["--task", task]
        run_command(task_command)
    if len(checkpoints) >= 2:
        comparison_dir = evaluation_dir / "checkpoint_comparison"
        command = [sys.executable, "scripts/compare_checkpoints.py", "--config", CONFIG_PATH, "--review-per-checkpoint", "50", "--output-dir", comparison_dir]
        for index, checkpoint in enumerate(checkpoints):
            command += ["--checkpoint", f"checkpoint_{index:02d}={checkpoint}"]
        run_command(command)
        llm_judge = comparison_dir / "llm_judge"
        print(f"LLM judge bundle: {llm_judge}")
        print("Attach judge_prompt.md and each blinded batch from llm_judge/batches to this Codex task. Save returned JSONL under llm_judge/results, then run scripts/score_story_judgments.py --key <review_key> --judgments <judgments_jsonl> --reviewer llm --comparison-summary <comparison_dir>/comparison_summary.json --output <comparison_dir>/llm_judge/results/scored_llm.json. Human review is optional.")
    else:
        print("One checkpoint evaluated. Preserve at least two checkpoints to create a blinded comparison bundle.")
else:
    print("Skipped: this cell acts only when RUN_STAGE='evaluate'.")

## 12. Review persisted evidence

In [ ]:
print("Drive root:", DRIVE_ROOT)
for path in sorted(EVIDENCE_DIR.glob("*")):
    print("evidence:", path.name, path.stat().st_size, "bytes")
if (RUN_DIR / "metrics.csv").is_file():
    print("metrics:", RUN_DIR / "metrics.csv")
if (RUN_DIR / "checkpoints/latest.pt").is_file():
    state = torch.load(RUN_DIR / "checkpoints/latest.pt", map_location="cpu", weights_only=False)["state"]
    print("latest checkpoint state:", {key: state.get(key) for key in ("global_step", "tokens_processed", "best_val_loss")})